# 04A Vector Autoregression: Estimation and Granger Causality

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/08-Time-Series/04A_VAR_Estimation_and_Granger.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=08-Time-Series/04A_VAR_Estimation_and_Granger.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


In [1]:
# === Environment Setup ===
import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas_datareader.data as web
    PDR_AVAILABLE = True
except ImportError:
    PDR_AVAILABLE = False

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, precision=4)


Environment initialized for Vector Autoregression (VAR) Models.


## The Lens: The Web of Macroeconomics
**What economic problem are we solving?**
The economy is a deeply interconnected system. A hike in interest rates doesn't just lower inflation; it might also slow GDP growth, increase unemployment, and strengthen the currency. These variables feed back into each other over time: lower growth might eventually lower inflation further, prompting the central bank to cut rates. Single-equation models (like ARIMA) ignore these feedback loops. To understand the **dynamic transmission** of shocks—how a surprise in one variable ripples through the entire economy—we need a model that treats all variables as endogenous.

**Why do we need this method?**
**Vector Autoregression (VAR)** is the workhorse model for this task. Introduced by Christopher Sims as a response to the "Incredible Restrictions" of large structural models, VARs let the data speak for themselves. By modeling a vector of variables as a function of their own lags and the lags of all other variables, VARs capture the complex dynamics of the macroeconomy without imposing heavy-handed theoretical assumptions. They allow us to answer questions like: *"If the Fed raises rates by 25 basis points today, what is the path of inflation over the next 24 months?"*

**Scope note:** Module **06-Econometrics/09_Classical_Time_Series_Analysis.ipynb** covers classical univariate tools (stationarity diagnostics, ARIMA, volatility). Module **08-Time-Series** focuses on multivariate dynamics, identification, and forecasting workflows (VARs, state-space, IRFs).

### Learning Objectives
* **Specify and estimate** a VAR(p) model and select the lag order using information criteria.
* **Conduct** Granger causality tests to assess predictive relationships between variables.
* **Compute and interpret** impulse response functions (IRFs) and forecast error variance decompositions (FEVDs).
* **Explain** the identification problem and the role of the Cholesky ordering assumption.

### Prerequisites
* **`06-Econometrics/09_Classical_Time_Series_Analysis.ipynb`**: Stationarity diagnostics and ARIMA foundations.
* **`02-Numerical-Methods/01_Linear_Algebra.ipynb`**: Matrix algebra and eigenvalues for stability conditions.
* **`01-Foundations/13_Pandas.ipynb`**: Time-indexed data manipulation.
* **Learning-path prerequisite:** [`03_ARIMA_and_Forecasting.ipynb`](03_ARIMA_and_Forecasting.ipynb)


> **Learning path:** Building on [`03_ARIMA_and_Forecasting.ipynb`](03_ARIMA_and_Forecasting.ipynb); next continue with [`04B_VAR_Identification_and_Structural_Shocks.ipynb`](04B_VAR_Identification_and_Structural_Shocks.ipynb).


### Table of Contents
1. [The Lens: The Web of Macroeconomics](#The-Lens:-The-Web-of-Macroeconomics)
2. [Introduction: From Univariate to Multivariate Time Series](#intro)
3. [The VAR(p) Model](#var-model)
   - [Estimation and Lag Selection](#estimation)
4. [Granger Causality](#granger)
5. [Summary](#summary)


<a id='intro'></a>
## 1. Introduction: From Univariate to Multivariate Time Series

The ARIMA models we have studied so far are **univariate**—they model a single time series based only on its own past. However, economic variables are interconnected. GDP growth is affected by monetary policy, inflation depends on unemployment, and so on. To model these rich dynamic relationships, we need a **multivariate** framework.

Prior to the 1980s, macroeconomic modeling was dominated by large-scale structural equation models. These models were heavily based on economic theory but were criticized by **Robert Lucas** in what became known as the **Lucas critique**. He argued that the parameters of these models were not stable under policy changes, as they failed to account for how rational agents would change their expectations and behavior. 

**Vector Autoregression (VAR)** models, introduced in a seminal 1980 paper by **Christopher Sims** (who later won a Nobel Prize for this work), were proposed as a direct response to this critique. A VAR is a more flexible, data-driven approach that imposes fewer theoretical restrictions. It treats every variable in the system as endogenous, creating a system of equations where each variable is regressed on its own lags and the lags of all other variables. This allows us to capture the complex feedback loops present in the economy and, crucially, to trace out how shocks to one variable dynamically affect all other variables in the system, with more transparent identifying assumptions.


<a id='var-model'></a>
## 2. The VAR(p) Model

A VAR(p) model with $k$ variables is a system of $k$ equations. For a simple two-variable ($y_1, y_2$) VAR(1) model, the system would be:
$$ y_{1,t} = c_1 + \phi_{11,1} y_{1,t-1} + \phi_{12,1} y_{2,t-1} + \epsilon_{1,t} $$ 
$$ y_{2,t} = c_2 + \phi_{21,1} y_{1,t-1} + \phi_{22,1} y_{2,t-1} + \epsilon_{2,t} $$ 
Here, the current value of $y_1$ depends on the first lag of both $y_1$ and $y_2$. The error terms $(\epsilon_{1,t}, \epsilon_{2,t})$ are assumed to be white noise, but they can be contemporaneously correlated with each other, $Cov(\epsilon_{1,t}, \epsilon_{2,t}) = \sigma_{12}$.

<a id='estimation'></a>
### Estimation and Lag Selection
Since the right-hand-side variables in each equation are all lagged (pre-determined), there are no endogeneity issues, and each equation can be estimated individually using OLS. The main specification choice is the **lag order (p)**. This is almost always chosen by estimating the VAR for a range of different lag lengths and selecting the one that minimizes an information criterion like the AIC or BIC.


<a id='var-lab'></a>
### 2.1 Code Lab: OLS Estimation of a VAR(1)

Each equation of a VAR(1) is estimated by OLS on the same regressors — the lagged levels. Stacking equations, $Y = B X + E$ where $X_t = [1, y_{1,t}, y_{2,t}]'$, the coefficient matrix is recovered jointly by least squares: $\hat{B} = Y X'(X X')^{-1}$. The lab below simulates a bivariate VAR(1) with **known** coefficients and checks that OLS recovers them.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# True DGP: y_t = c + A y_{t-1} + e_t,  A known
c_true = np.array([0.5, -0.2])
A_true = np.array([[0.6, 0.1],
                   [0.0, 0.4]])
Sigma = np.array([[1.0, 0.3],
                  [0.3, 1.0]])
L = np.linalg.cholesky(Sigma)

T = 2000
y = np.zeros((T, 2))
for t in range(1, T):
    y[t] = c_true + A_true @ y[t - 1] + L @ rng.standard_normal(2)

# --- OLS estimation: regress y_t on [1, y_{t-1}] ---
Y = y[1:]                      # dependent variables, shape (T-1, 2)
X = np.column_stack([np.ones(T - 1), y[:-1]])   # regressors, shape (T-1, 3)
B_hat = np.linalg.lstsq(X, Y, rcond=None)[0]    # (3, 2): rows [const, A_11 A_21; ...]

A_hat = B_hat[1:].T            # first row of B is the constant
c_hat = B_hat[0]

print("true A:\n", A_true, "\nestimated A:\n", np.round(A_hat, 3))
print("true c:", c_true, " estimated c:", np.round(c_hat, 3))
assert np.allclose(A_hat, A_true, atol=0.05), "OLS should recover A closely"
assert np.allclose(c_hat, c_true, atol=0.10)
print("OLS recovers the true VAR(1) coefficients.")


<a id='granger'></a>
## 3. Granger Causality

A key question in a VAR is whether one variable is useful for forecasting another. The **Granger causality test** provides a statistical answer to this. The principle is:

> Variable $X$ is said to **Granger-cause** variable $Y$ if past values of $X$ contain information that helps predict future values of $Y$, even after accounting for the information contained in past values of $Y$ itself.

In the context of our two-variable VAR(1) above, we would test if $y_2$ Granger-causes $y_1$ by performing an F-test on the null hypothesis that the coefficient $\phi_{12,1}$ is zero. If we reject the null, it means that past values of $y_2$ have statistically significant predictive power for $y_1$.


<a id='granger-lab'></a>
### 3.1 Code Lab: A Granger Causality F-Test by Hand

$y_2$ does **not** Granger-cause $y_1$ if the lags of $y_2$ add no explanatory power in the $y_1$ equation. Test with an F-statistic comparing restricted (own lags only) and unrestricted (both lags) OLS fits:

$$
F = \frac{(RSS_r - RSS_u)/q}{RSS_u/(T - k_u)} \sim F(q,\, T-k_u).
$$


In [ ]:
from scipy import stats

# Simulate a system where y2 Granger-causes y1 but NOT vice versa:
#   y1_t = 0.5 y1_{t-1} + 0.4 y2_{t-1} + e1      <- y2's lag matters
#   y2_t = 0.7 y2_{t-1} + e2                     <- y1's lag absent
rng = np.random.default_rng(7)
T = 1000
y = np.zeros((T, 2))
for t in range(1, T):
    y[t, 0] = 0.5 * y[t - 1, 0] + 0.4 * y[t - 1, 1] + rng.standard_normal()
    y[t, 1] = 0.7 * y[t - 1, 1] + rng.standard_normal()

def ols_rss(dep, lags):
    """RSS of regressing dep on a constant plus `lags` lag columns."""
    X = np.column_stack([np.ones(len(dep))] + lags)
    b, *_ = np.linalg.lstsq(X, dep, rcond=None)
    return float(np.sum((dep - X @ b) ** 2)), X.shape[1]

y1, y2 = y[1:, 0], y[:-1, 0]
lag1_y1, lag1_y2 = y[:-1, 0], y[:-1, 1]

rss_u, k_u = ols_rss(y1, [lag1_y1, lag1_y2])       # unrestricted
rss_r, k_r = ols_rss(y1, [lag1_y1])                 # restricted (drop y2's lag)
q = k_u - k_r
F = ((rss_r - rss_u) / q) / (rss_u / (len(y1) - k_u))
p_value = 1.0 - stats.f.cdf(F, q, len(y1) - k_u)
print(f"Granger F({q}, {len(y1)-k_u}) = {F:.2f}, p = {p_value:.2e}")
assert p_value < 0.01, "y2 should strongly Granger-cause y1"

# Reverse direction: y1 should NOT Granger-cause y2
rss_u2, k_u2 = ols_rss(y[1:, 1], [y[:-1, 1], y[:-1, 0]])
rss_r2, _ = ols_rss(y[1:, 1], [y[:-1, 1]])
F2 = ((rss_r2 - rss_u2) / (k_u2 - 1)) / (rss_u2 / (len(y) - 1 - k_u2))
p2 = 1.0 - stats.f.cdf(F2, 1, len(y) - 1 - k_u2)
print(f"Reverse F = {F2:.2f}, p = {p2:.3f}  (should NOT reject at 5%)")
assert p2 > 0.05
print("Directionality detected correctly.")


## Exercises

**1. Mechanism and assumptions (Conceptual):** For **04A Vector Autoregression: Estimation and Granger Causality**, identify the stochastic assumptions that make the model estimable and state how stationarity, invertibility, or identification can be checked from the fitted object.

**2. Reproduce and diagnose (Applied):** Fit the method covered in 1. Introduction: From Univariate to Multivariate Time Series, 2. The VAR(p) Model to a time-ordered series. Diagnose residual dependence and stability, then evaluate a rolling or expanding-window out-of-sample forecast against a naive baseline.

**3. Robust extension (Challenge):** Alter one structural restriction, lag/order choice, or innovation distribution. Explain how impulse responses, forecasts, or uncertainty change and whether the conclusion survives the alternative specification.

<details>
<summary>Solution guidance</summary>

A strong solution states assumptions before computation, includes an independent diagnostic or limiting-case check, and interprets the result in the units of the economic problem. For the challenge, separate changes caused by the economic assumption from changes caused by numerical approximation or tuning.

</details>


# Summary

This part introduced VAR estimation, lag selection, and Granger causality as tools for modeling multivariate time series dynamics.


## References & Further Reading

- Hamilton, J. D. (1994). *Time Series Analysis*. Princeton University Press.
- Lütkepohl, H. (2005). *New Introduction to Multiple Time Series Analysis*. Springer.
- Hyndman, R. J. & Athanasopoulos, G. *Forecasting: Principles and Practice* (3rd ed.). OTexts.
